# MPD Colab GPU — BPR-MF + Context 재랭킹

실험 런타임은 **Google Colab GPU**다. 로컬 CPU는 쓰지 않는다.

시작하기 전에 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. 무료 T4(16GB)를 기본으로 보고, 학습은 `--cpu-interactions`를 켠다.

1. BPR-MF 학습 (`src/bpr_mf.py`) → Drive에 체크포인트
2. 같은 체크포인트로 재랭킹 (`src/context_rerank.py`): B0–M3, T0, C1, X1, P1, 후보 N=500

체크포인트가 이미 있으면 학습 셀은 건너뛴다. Drive의 `src/context_rerank.py`를 이 버전으로 다시 올린 뒤 재랭킹 셀만 실행한다.

In [ ]:
# Colab에 설치된 PyTorch가 GPU를 정상적으로 인식하는지 확인한다.
import torch

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임 유형을 GPU로 변경한 뒤 다시 실행하세요."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 프로젝트, MPD 데이터, 학습 결과를 사용할 Google Drive를 연결한다.
from google.colab import drive

drive.mount("/content/drive")

## 경로 설정

배포 폴더와 MPD 데이터를 Google Drive에 올린 뒤, 실제 폴더 위치에 맞게 아래 경로를 수정하세요.

In [ ]:
# 본인의 Google Drive 폴더 구조에 맞게 세 경로를 수정한다.
# PROJECT_DIR 에는 src/bpr_mf.py, src/context_rerank.py 가 있어야 한다.
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BOAZ_mini_project01")
DATA_DIR = Path("/content/drive/MyDrive/mpd/data")
OUTPUT_DIR = Path("/content/drive/MyDrive/bpr_mf_outputs")

SCRIPT_PATH = PROJECT_DIR / "src/bpr_mf.py"
assert SCRIPT_PATH.is_file(), f"실행 파일을 찾을 수 없습니다: {SCRIPT_PATH}"
assert (PROJECT_DIR / "src/context_rerank.py").is_file(), "src/context_rerank.py 가 필요합니다."
assert DATA_DIR.is_dir(), f"데이터 폴더를 찾을 수 없습니다: {DATA_DIR}"
slice_count = len(list(DATA_DIR.glob("mpd.slice.*.json")))
assert slice_count > 0, "MPD JSON slice 파일을 찾을 수 없습니다."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("실행 파일:", SCRIPT_PATH)
print("MPD slice 수:", slice_count)
print("결과 저장 폴더:", OUTPUT_DIR)

## 학습 설정

스펙 원점은 `MAX_SLICES = 10`, `EPOCHS = 10`이다. 연결 확인만 할 때는 `1 / 1`로 줄인다. `MAX_SLICES = 0`이면 전체 MPD이며 1차 실험 범위가 아니다.

T4에서 VRAM이 부족하면 `CPU_INTERACTIONS = True`를 유지하고 `BATCH_SIZE`만 낮춘다.

In [ ]:
# 실험 규모와 BPR-MF 하이퍼파라미터를 설정한다.
MAX_SLICES = 10
EPOCHS = 10
BATCH_SIZE = 8192
FACTORS = 64
LEARNING_RATE = 0.01
REGULARIZATION = 0.0025
EVAL_NEGATIVES = 100
SEED = 42

# True이면 interaction을 CPU에 보관해 GPU 메모리 사용량을 줄인다.
CPU_INTERACTIONS = True

# None이면 epoch마다 모든 학습 interaction 수만큼 샘플링한다.
SAMPLES_PER_EPOCH = None

OUTPUT_PATH = OUTPUT_DIR / f"bpr_mf_slices_{MAX_SLICES}.pt"
print("체크포인트 저장 경로:", OUTPUT_PATH)

## 학습 및 평가

학습이 끝나면 validation/test의 HitRate@K, NDCG@K, sampled AUC가 출력되고 체크포인트가 Google Drive에 저장됩니다.

In [ ]:
# 설정값으로 실행 명령을 구성하고 학습 프로세스를 시작한다.
import subprocess
import sys

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--data-dir", str(DATA_DIR),
    "--output", str(OUTPUT_PATH),
    "--max-slices", str(MAX_SLICES),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--factors", str(FACTORS),
    "--lr", str(LEARNING_RATE),
    "--reg", str(REGULARIZATION),
    "--eval-negatives", str(EVAL_NEGATIVES),
    "--seed", str(SEED),
]

if CPU_INTERACTIONS:
    command.append("--cpu-interactions")
if SAMPLES_PER_EPOCH is not None:
    command.extend(["--samples-per-epoch", str(SAMPLES_PER_EPOCH)])

print("실행 명령:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
# 저장된 체크포인트의 기본 정보와 평가 결과를 확인한다.
checkpoint = torch.load(OUTPUT_PATH, map_location="cpu", weights_only=False)

print("저장 파일:", OUTPUT_PATH)
print("플레이리스트 수:", len(checkpoint["playlist_ids"]))
print("트랙 수:", len(checkpoint["track_uris"]))
print("Validation:", checkpoint["validation_metrics"])
print("Test:", checkpoint["test_metrics"])

## Context 재랭킹 (B0 → C1, X1, P1)

학습이 끝난 체크포인트를 재사용한다. 학습 셀은 건너뛰면 된다. Drive에 `src/context_rerank.py`를 다시 올린 뒤 이 셀만 실행한다.

이번 실행은 X1/P1을 **실제로 켠다**.

1. 후보 `N=500`. 결과는 `*_x1p1_on.json` (이전 λ=0 실행을 덮지 않음)
2. X1: C1과 **같은 λ**에서 공식만 바꿈. Highly niche는 인기곡 항 유지, Discovery는 일괄 하향 금지
3. P1: 제목 CSV가 **반드시** 있어야 한다. Lift λ=0은 제외하고 고른다

Drive 경로 예:

`PROJECT_DIR/purpose_titles/outputs/all_unique_purpose_labels.csv`

In [ ]:
# BPR 체크포인트 위에서 gpt.md 실험 B0–M3 + T0 + C1 + X1 + P1를 Colab GPU로 돌린다.
# X1/P1이 λ=0으로 접히지 않게, 제목 CSV가 있을 때만 이 셀을 실행한다.
import os
import subprocess
import sys

assert torch.cuda.is_available(), "재랭킹도 GPU 런타임에서 실행하세요."
PURPOSE_LABELS = PROJECT_DIR / "purpose_titles/outputs/all_unique_purpose_labels.csv"
assert PURPOSE_LABELS.is_file(), (
    f"P1용 제목 라벨이 없습니다: {PURPOSE_LABELS}\n"
    "Drive PROJECT_DIR 아래 purpose_titles/outputs/all_unique_purpose_labels.csv 를 올리세요."
)
CANDIDATE_N = 500
RERANK_OUTPUT = OUTPUT_DIR / f"{OUTPUT_PATH.stem}_context_rerank_n{CANDIDATE_N}_x1p1_on.json"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
score_batch = 128 if vram_gb < 20 else 256

command = [
    sys.executable,
    "-m",
    "src.context_rerank",
    "--checkpoint", str(OUTPUT_PATH),
    "--data-dir", str(DATA_DIR),
    "--candidate-n", str(CANDIDATE_N),
    "--top-k", "10",
    "--device", "cuda",
    "--score-batch-size", str(score_batch),
    "--purpose-labels", str(PURPOSE_LABELS),
    "--output", str(RERANK_OUTPUT),
]

print("GPU:", torch.cuda.get_device_name(0), f"VRAM={vram_gb:.1f}GB", "score_batch", score_batch)
print("purpose labels:", PURPOSE_LABELS)
print("실행 명령:", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, env={**os.environ, "PYTHONPATH": str(PROJECT_DIR)}, check=True)
print("결과:", RERANK_OUTPUT)

## 중간점검 베스트 추천 (BEST-X1)

1차 실험에서 **순위(NDCG@10)가 가장 높았던 설정만** 고정한다. 학습은 다시 하지 않는다.

- 점수 = `z(BPR) + 1.0 × z(X1 프로필)`
- X1 = C1 연속 프로필 + Discovery 플리에는 인기곡 일괄 하향 금지
- **빼는 것:** M1(꺼짐), M2(탐색형 파괴), P1(목적 칸 NDCG 하락)

Ablation(`context_rerank.py`)이 아니라 `src/best_rerank.py`다. 목적 CSV는 필요 없다.
체크포인트와 MPD slice는 위와 같다. Drive에 `src/best_rerank.py`를 올린 뒤 이 셀만 실행한다.


In [ ]:
# 실험에서 이긴 X1만 고정해 중간점검 베스트 추천을 만든다.
# 학습 셀은 건너뛰면 된다. Drive에 src/best_rerank.py 가 있어야 한다.
import os
import subprocess
import sys

assert torch.cuda.is_available(), "베스트 재랭킹도 GPU 런타임에서 실행하세요."
assert (PROJECT_DIR / "src/best_rerank.py").is_file(), "src/best_rerank.py 가 필요합니다."
assert OUTPUT_PATH.is_file(), f"체크포인트가 없습니다: {OUTPUT_PATH}"

CANDIDATE_N = 500
LAMBDA_CTX = 1.0
BEST_OUTPUT = OUTPUT_DIR / f"{OUTPUT_PATH.stem}_best_rerank_n{CANDIDATE_N}.json"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
score_batch = 128 if vram_gb < 20 else 256

command = [
    sys.executable,
    "-m",
    "src.best_rerank",
    "--checkpoint", str(OUTPUT_PATH),
    "--data-dir", str(DATA_DIR),
    "--candidate-n", str(CANDIDATE_N),
    "--lambda-ctx", str(LAMBDA_CTX),
    "--top-k", "10",
    "--device", "cuda",
    "--score-batch-size", str(score_batch),
    "--output", str(BEST_OUTPUT),
    "--write-topk",
]

print("GPU:", torch.cuda.get_device_name(0), f"VRAM={vram_gb:.1f}GB", "score_batch", score_batch)
print("실행 명령:", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, env={**os.environ, "PYTHONPATH": str(PROJECT_DIR)}, check=True)
print("결과:", BEST_OUTPUT)
print("top-K:", BEST_OUTPUT.with_name(BEST_OUTPUT.stem + "_topk.jsonl"))


## 유형별 후보 (TYPED-X1)

중간점검 BEST-X1은 그대로 둔다. 이 셀은 **다음 실험**: BPR 500을 전 플리에 늘리지 않고, 팬형·탐색형만 하위 칸을 갈아 끼운다.

- 팬형: 아는 가수의 미수록곡
- 탐색형: 이미 있는 가수 여러 명 커버
- 나머지: BPR 500 유지
- 그 다음 점수는 기존 X1

Drive에 `src/typed_candidates.py`를 올린 뒤 GPU에서 실행. 학습은 다시 하지 않는다.
비교: B0 / BEST-X1 / TYPED-B0 / TYPED-X1. `candidate_recall`이 팬형에서 오르는지가 핵심이다.

In [ ]:
# 유형별 후보 + X1. 학습 셀은 건너뛴다. Drive에 src/typed_candidates.py 가 있어야 한다.
import os
import subprocess
import sys

assert torch.cuda.is_available(), "GPU 런타임에서 실행하세요."
assert (PROJECT_DIR / "src/typed_candidates.py").is_file(), "src/typed_candidates.py 가 필요합니다."
assert OUTPUT_PATH.is_file(), f"체크포인트가 없습니다: {OUTPUT_PATH}"

CANDIDATE_N = 500
INJECT_K = 80
TYPED_OUTPUT = OUTPUT_DIR / f"{OUTPUT_PATH.stem}_typed_cand_n{CANDIDATE_N}_inj{INJECT_K}.json"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
score_batch = 128 if vram_gb < 20 else 256

command = [
    sys.executable, "-m", "src.typed_candidates",
    "--checkpoint", str(OUTPUT_PATH),
    "--data-dir", str(DATA_DIR),
    "--candidate-n", str(CANDIDATE_N),
    "--inject-k", str(INJECT_K),
    "--lambda-ctx", "1.0",
    "--top-k", "10",
    "--device", "cuda",
    "--score-batch-size", str(score_batch),
    "--output", str(TYPED_OUTPUT),
]
print("GPU:", torch.cuda.get_device_name(0), f"VRAM={vram_gb:.1f}GB")
print("실행:", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, env={**os.environ, "PYTHONPATH": str(PROJECT_DIR)}, check=True)
print("결과:", TYPED_OUTPUT)

## 팬형만 후보 (FAN-X1)

바로 위 TYPED는 탐색형까지 바꿨고, 탐색형·Q1 탐색형이 깨졌다. 이 셀은 **팬형만** 하위 칸을 아는 가수 곡으로 바꾼다. 탐색형·나머지는 BEST-X1과 같은 BPR 500.

같은 `.pt`, 같은 X1, 학습 없음. `src/typed_candidates.py`를 Drive에 **다시 올린 뒤** 실행.

볼 것: 팬형 NDCG/candidate_recall이 이전 TYPED만큼 사는지, 탐색형·Q1 탐색형이 BEST-X1과 같은지.

In [ ]:
# 팬형만 후보 교체 + X1. 학습 셀은 건너뛴다. Drive의 typed_candidates.py 를 이 버전으로 덮어쓴 뒤 실행.
import os
import subprocess
import sys

assert torch.cuda.is_available(), "GPU 런타임에서 실행하세요."
assert (PROJECT_DIR / "src/typed_candidates.py").is_file(), "src/typed_candidates.py 가 필요합니다."
assert OUTPUT_PATH.is_file(), f"체크포인트가 없습니다: {OUTPUT_PATH}"

CANDIDATE_N = 500
INJECT_K = 80
FAN_OUTPUT = OUTPUT_DIR / f"{OUTPUT_PATH.stem}_typed_cand_n{CANDIDATE_N}_inj{INJECT_K}_centric.json"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
score_batch = 128 if vram_gb < 20 else 256

command = [
    sys.executable, "-m", "src.typed_candidates",
    "--checkpoint", str(OUTPUT_PATH),
    "--data-dir", str(DATA_DIR),
    "--candidate-n", str(CANDIDATE_N),
    "--inject-k", str(INJECT_K),
    "--lambda-ctx", "1.0",
    "--top-k", "10",
    "--device", "cuda",
    "--score-batch-size", str(score_batch),
    "--centric-only",
    "--output", str(FAN_OUTPUT),
]
print("GPU:", torch.cuda.get_device_name(0), f"VRAM={vram_gb:.1f}GB")
print("실행:", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, env={**os.environ, "PYTHONPATH": str(PROJECT_DIR)}, check=True)
print("결과:", FAN_OUTPUT)